# 02 · Calidad y gobierno de datos (R2)
Parte leyendo la capa limpia que deja `01_ingesta` (`data/clean/covid_limpio.parquet`).

| Historia | Sección |
|---|---|
| H2.1 · De acumulado a diario | 1 |
| H2.2 · Reglas de calidad (6 dimensiones) | 2 |
| H2.3 · Cuantificar los problemas | 3 |
| H2.4 · Índice de calidad por país y decisiones | 4 |
| H2.6 · Linaje | 5 |

In [1]:
import pandas as pd

df = pd.read_parquet("../data/clean/covid_limpio.parquet")
print("Filas:", len(df), "| columnas:", df.shape[1])

Filas: 4287473 | columnas: 19


## 1 · De acumulado a diario (H2.1)
Los reportes traen **acumulados**: los casos nuevos se calculan restando el día anterior **del mismo territorio**
(país + provincia + condado). Los acumulados que retroceden se **marcan, no se borran** (PPT S14, lámina 27).

### Filas repetidas
Entre el 23-ene y el 22-mar-2020 hay territorios con dos filas el mismo día: casi siempre una trae el dato
y la otra un 0 (por ejemplo, Bahamas 3 y 0). Si no se resuelven, la resta del día anterior da cualquier cosa.
Dejamos una fila por territorio y día: la de mayor `confirmed`.

In [2]:
llave = ["country_region", "province_state", "admin2"]

repetidas = df.duplicated(llave + ["fecha_reporte"], keep=False)
print("Filas repetidas (territorio × día):", repetidas.sum())

df = (df.sort_values("confirmed", ascending=False)        # la fila con el dato queda primero
        .drop_duplicates(llave + ["fecha_reporte"])        # y se descarta la otra
        .sort_values(llave + ["fecha_reporte"])
        .reset_index(drop=True))
print("Filas después:", len(df))

Filas repetidas (territorio × día): 156


Filas después: 4287395


### Nivel territorio
- El primer día de un territorio que aparece **junto con su país** vale su acumulado (antes tenía 0 casos).
- Si aparece **después** que su país, nace por un cambio de granularidad (condados de EE.UU. desde el 22-mar-2020,
  regiones de Chile desde el 20-may-2020): su primer acumulado arrastra casos anteriores.
  Lo dejamos vacío (`NaN`) y lo marcamos en `primer_dia_por_cambio`. Imputar es inventar.

In [3]:
g = df.groupby(llave)
df["casos_nuevos"] = g["confirmed"].diff()
df["fallecidos_nuevos"] = g["deaths"].diff()

primer_dia = g.cumcount() == 0
inicio_pais = df.groupby("country_region")["fecha_reporte"].transform("min")
df["primer_dia_por_cambio"] = primer_dia & (df["fecha_reporte"] > inicio_pais)

nace_con_pais = primer_dia & ~df["primer_dia_por_cambio"]
df.loc[nace_con_pais, "casos_nuevos"] = df.loc[nace_con_pais, "confirmed"]
df.loc[nace_con_pais, "fallecidos_nuevos"] = df.loc[nace_con_pais, "deaths"]

# Marcar, no borrar
df["retrocede_casos"] = df["casos_nuevos"] < 0
df["retrocede_fallecidos"] = df["fallecidos_nuevos"] < 0

print("Territorios:", primer_dia.sum(), "| nacen por cambio de granularidad:", df["primer_dia_por_cambio"].sum())
print("Filas con casos que retroceden:", df["retrocede_casos"].sum(),
      "| con fallecidos que retroceden:", df["retrocede_fallecidos"].sum())

Territorios: 4296 | nacen por cambio de granularidad: 4038
Filas con casos que retroceden: 44587 | con fallecidos que retroceden: 16186


### Nivel país
Para un país primero sumamos todos sus territorios por día y **después** restamos el día anterior.
Así el paso de Chile nacional → 16 regiones no produce saltos.

In [4]:
pais = (
    df.groupby(["country_region", "fecha_reporte"], as_index=False)[["confirmed", "deaths"]]
      .sum(min_count=1)                          # min_count=1: si todo es NaN, queda NaN (no 0)
)
gp = pais.groupby("country_region")
pais["casos_nuevos"] = gp["confirmed"].diff()
pais["fallecidos_nuevos"] = gp["deaths"].diff()

primer_dia_pais = gp.cumcount() == 0
pais.loc[primer_dia_pais, "casos_nuevos"] = pais.loc[primer_dia_pais, "confirmed"]
pais.loc[primer_dia_pais, "fallecidos_nuevos"] = pais.loc[primer_dia_pais, "deaths"]

pais["retrocede_casos"] = pais["casos_nuevos"] < 0
pais["retrocede_fallecidos"] = pais["fallecidos_nuevos"] < 0
pais.head()

,country_region,fecha_reporte,confirmed,deaths,casos_nuevos,fallecidos_nuevos,retrocede_casos,retrocede_fallecidos
0,Afghanistan,2020-02-24,5.0,0.0,5.0,0.0,False,False
1,Afghanistan,2020-02-25,5.0,0.0,0.0,0.0,False,False
2,Afghanistan,2020-02-26,5.0,0.0,0.0,0.0,False,False
3,Afghanistan,2020-02-27,5.0,0.0,0.0,0.0,False,False
4,Afghanistan,2020-02-28,5.0,0.0,0.0,0.0,False,False


### Controles

In [5]:
chile = pais[pais["country_region"] == "Chile"]
print("Chile · suma de casos nuevos:", int(chile["casos_nuevos"].sum()),
      "| último acumulado:", int(chile["confirmed"].iloc[-1]))
print("Chile · suma de fallecidos nuevos:", int(chile["fallecidos_nuevos"].sum()),
      "| último acumulado:", int(chile["deaths"].iloc[-1]))

print("\nPaíses:", pais["country_region"].nunique())
print("Países con algún día en que los casos retroceden:", pais.loc[pais["retrocede_casos"], "country_region"].nunique())
print("Países con algún día en que los fallecidos retroceden:", pais.loc[pais["retrocede_fallecidos"], "country_region"].nunique())

Chile · suma de casos nuevos: 5192286 | último acumulado: 5192286
Chile · suma de fallecidos nuevos: 64273 | último acumulado: 64273

Países: 201
Países con algún día en que los casos retroceden: 159
Países con algún día en que los fallecidos retroceden: 111


## 2 · Reglas de calidad en las seis dimensiones (H2.2)
Un control que cabe en una pantalla (PPT S8, láminas 16 y 26): cada regla es una tupla
`(dimensión, regla, filas que FALLAN, umbral %, crítica)` y `reglas_calidad(df)` devuelve una tabla con su estado.

- **OK**: el % de filas que fallan no supera el umbral.
- **RECHAZA**: supera el umbral y la regla es crítica (el dato no se puede usar así).
- **ALERTA**: supera el umbral pero la regla no es crítica (se marca y se sigue).

El % se compara **sin redondear**: con umbral 0 %, una sola fila mala ya falla.

Se corre sobre la tabla que deja la sección 1, porque la exactitud usa `retrocede_casos` y `retrocede_fallecidos`.
Por eso la unicidad da 0: las filas repetidas ya se resolvieron ahí. **En el crudo eran 156 filas (RECHAZA).**

Sobre la oportunidad: `last_update` suele ser el día **siguiente** a `fecha_reporte` (el reporte de un día se publica
a la mañana siguiente), así que una diferencia de −1 es lo normal. Falla si el dato lleva más de 7 días sin actualizarse.

In [6]:
def reglas_calidad(df):
    dias_atraso = (df["fecha_reporte"] - df["last_update"]).dt.days
    llave_dia = ["country_region", "province_state", "admin2", "fecha_reporte"]

    R = [
        # (dimensión,     regla,                                  filas que FALLAN,                                       umbral %, crítica)
        ("Completitud",  "confirmed ausente",                     df["confirmed"].isna(),                                  1.0,     True),
        ("Completitud",  "territorio sin uid (no cruza con UID)", df["uid"].isna(),                                        1.0,     False),
        ("Unicidad",     "territorio × día repetido",             df.duplicated(llave_dia, keep=False),                    0.0,     True),
        ("Validez",      "confirmed o deaths negativo",           (df["confirmed"] < 0) | (df["deaths"] < 0),              0.0,     True),
        ("Validez",      "lat o long fuera de rango",             (df["lat"].abs() > 90) | (df["long"].abs() > 180),       0.0,     True),
        ("Validez",      "fecha_reporte fuera de 22-ene-2020 → 9-mar-2023",
                                                                  ~df["fecha_reporte"].between("2020-01-22", "2023-03-09"), 0.0,    True),
        ("Consistencia", "fallecidos mayores que casos",          df["deaths"] > df["confirmed"],                          0.1,     False),
        ("Exactitud",    "acumulado de casos retrocede",          df["retrocede_casos"],                                   1.0,     False),
        ("Exactitud",    "acumulado de fallecidos retrocede",     df["retrocede_fallecidos"],                              1.0,     False),
        ("Oportunidad",  "last_update con más de 7 días de atraso", dias_atraso > 7,                                       5.0,     False),
    ]

    filas = []
    for dimension, regla, falla, umbral, critica in R:
        n = int(falla.sum())
        pct = 100 * n / len(df)
        if pct <= umbral:
            estado = "OK"
        elif critica:
            estado = "RECHAZA"
        else:
            estado = "ALERTA"
        filas.append([dimension, regla, n, round(pct, 2), umbral, estado])

    return pd.DataFrame(filas, columns=["dimensión", "regla", "filas", "%", "umbral %", "estado"])


calidad = reglas_calidad(df)
calidad

,dimensión,regla,filas,%,umbral %,estado
0,Completitud,confirmed ausente,28,0.00,1.0,OK
1,Completitud,territorio sin uid (no cruza con UID),3159,0.07,1.0,OK
2,Unicidad,territorio × día repetido,0,0.00,0.0,OK
3,Validez,confirmed o deaths negativo,6,0.00,0.0,RECHAZA
4,Validez,lat o long fuera de rango,0,0.00,0.0,OK
5,Validez,fecha_reporte fuera de 22-ene-2020 → 9-mar-2023,0,0.00,0.0,OK
6,Consistencia,fallecidos mayores que casos,10777,0.25,0.1,ALERTA
7,Exactitud,acumulado de casos retrocede,44587,1.04,1.0,ALERTA
8,Exactitud,acumulado de fallecidos retrocede,16186,0.38,1.0,OK
9,Oportunidad,last_update con más de 7 días de atraso,15150,0.35,5.0,OK


## 3 · Cuantificar los problemas (H2.3)
Los cinco problemas que pide el enunciado, medidos sobre la tabla `pais` de la sección 1 (país × día).
Cada uno deja **un número** y su **top 10 de países**; al final, una tabla resumen.
Aquí solo se mide: qué hacer con cada problema se decide en H2.4.

In [7]:
resumen = []   # (problema, número, países afectados)

### 3.1 · Acumulado que baja
Días país en que el acumulado es menor que el del día anterior (`retrocede_casos` / `retrocede_fallecidos` de la sección 1).

In [8]:
for col, nombre in [("retrocede_casos", "casos"), ("retrocede_fallecidos", "fallecidos")]:
    malos = pais[pais[col]]
    resumen.append([f"Acumulado de {nombre} que baja (días país)", len(malos), malos["country_region"].nunique()])
    print(f"Días país con acumulado de {nombre} que baja:", len(malos), "| países:", malos["country_region"].nunique())

pais[pais["retrocede_casos"]].groupby("country_region").size().nlargest(10).rename("días que baja")

Días país con acumulado de casos que baja: 350 | países: 159
Días país con acumulado de fallecidos que baja: 272 | países: 111


country_region
Germany           46
Peru              11
Ecuador            6
France             6
Japan              6
Philippines        6
Spain              6
United Kingdom     6
Australia          5
Austria            5
Name: días que baja, dtype: int64

### 3.2 · Días sin reporte
Desde su primer caso, un día con el acumulado **igual** al del día anterior es un día en que el país no informó nada nuevo.
Ojo: un país chico que de verdad tuvo 0 casos se ve igual; con estos datos no se pueden separar.

También contamos los días en que el país **ni siquiera aparece** en el archivo del día.

In [9]:
con_casos = pais["confirmed"] > 0
sin_reporte = con_casos & (pais["casos_nuevos"] == 0)
resumen.append(["Días sin reporte (acumulado sin cambio)", sin_reporte.sum(), pais.loc[sin_reporte, "country_region"].nunique()])
print("Días país sin reporte:", sin_reporte.sum(), "| países:", pais.loc[sin_reporte, "country_region"].nunique())
print("\nPor año (en 2022 varios países pasan a informar una vez por semana):")
print(sin_reporte.groupby(pais["fecha_reporte"].dt.year).sum())

# Días en que el país falta en el archivo: desde su primera aparición hasta el 9-mar-2023
dias = pais.groupby("country_region")["fecha_reporte"].agg(["min", "count"])
dias["faltan"] = (pd.Timestamp("2023-03-09") - dias["min"]).dt.days + 1 - dias["count"]
print("\nDías país ausentes del archivo:", dias["faltan"].sum(), "| países:", (dias["faltan"] > 0).sum())
resumen.append(["Días en que el país falta en el archivo", dias["faltan"].sum(), (dias["faltan"] > 0).sum()])

# Top 10: % de sus días (desde el primer caso) sin reporte
(sin_reporte[con_casos].groupby(pais["country_region"]).mean() * 100).round(1).nlargest(10).rename("% días sin reporte")

Días país sin reporte: 69604 | países: 200

Por año (en 2022 varios países pasan a informar una vez por semana):
fecha_reporte
2020    13440
2021    14823
2022    32390
2023     8951
dtype: int64

Días país ausentes del archivo: 175 | países: 10


country_region
Antarctica              99.8
MS Zaandam              99.8
Korea, North            99.7
Holy See                98.8
Diamond Princess        98.6
Tuvalu                  95.2
Summer Olympics 2020    93.8
Marshall Islands        93.3
Micronesia              92.0
Winter Olympics 2022    91.7
Name: % días sin reporte, dtype: float64

### 3.3 · Picos por acumulación de rezagos
Un pico es un día con **más de 5 veces** la media de los 7 días anteriores y al menos 100 casos
(el mínimo evita marcar países chicos que pasan de 2 a 15 casos). Suele aparecer después de días sin reporte.
Control: Chile el 17-jun-2020 (36.179 casos, con rezagos sumados; ver `00_exploracion`) tiene que salir marcado.

In [10]:
media_7_previa = pais.groupby("country_region")["casos_nuevos"].transform(lambda s: s.shift(1).rolling(7).mean())
pais["pico_rezago"] = (pais["casos_nuevos"] > 5 * media_7_previa) & (pais["casos_nuevos"] >= 100)

picos = pais[pais["pico_rezago"]]
resumen.append(["Picos por acumulación de rezagos", len(picos), picos["country_region"].nunique()])
print("Picos:", len(picos), "| países:", picos["country_region"].nunique())
print("\nChile:")
print(picos.loc[picos["country_region"] == "Chile", ["fecha_reporte", "casos_nuevos"]].to_string(index=False))

picos.groupby("country_region").size().nlargest(10).rename("picos")

Picos: 2799 | países: 181

Chile:
fecha_reporte  casos_nuevos
   2020-03-20         196.0
   2020-06-17       36179.0
   2022-09-20       20038.0


country_region
Botswana               69
Cameroon               69
Mauritius              58
Sweden                 56
Costa Rica             50
Switzerland            49
Ecuador                48
United Kingdom         45
Congo (Brazzaville)    42
Cyprus                 42
Name: picos, dtype: int64

### 3.4 · Fallecidos mayores que casos
Se mide en dos niveles: filas de territorio (las mismas de la regla de consistencia de H2.2) y días país.
En territorio, la mayoría son filas `Unassigned` u `Out of ...`: fallecidos que se informan sin asignar a un condado
que no tiene casos asignados.

In [11]:
f = df["deaths"] > df["confirmed"]
resumen.append(["Fallecidos > casos (filas de territorio)", f.sum(), df.loc[f, "country_region"].nunique()])
print("Filas de territorio:", f.sum(), "| países:", df.loc[f, "country_region"].nunique())
print("admin2 más frecuentes:", df.loc[f, "admin2"].value_counts().head(3).to_dict())

fp = pais["deaths"] > pais["confirmed"]
resumen.append(["Fallecidos > casos (días país)", fp.sum(), pais.loc[fp, "country_region"].nunique()])
print("Días país:", fp.sum(), "| países:", pais.loc[fp, "country_region"].unique().tolist())

df[f].groupby("country_region").size().nlargest(10).rename("filas")

Filas de territorio: 10777 | países: 10
admin2 más frecuentes: {'Unassigned': 6085, '': 3652, 'Out of NY': 899}
Días país: 302 | países: ['Korea, North', 'Sudan', 'United Kingdom']


country_region
US                7125
United Kingdom    1377
Canada            1039
Colombia           508
Belgium            348
Korea, North       300
Netherlands         64
Peru                13
Chile                2
Sudan                1
Name: filas, dtype: int64

### 3.5 · Series vs suma de los reportes diarios
Las series globales (formato ancho, una columna por día) se leen desde raw, se pasan a formato largo,
se suman por país y se cruzan con `pais` por país y fecha.

Esto explica el **84 de la PPT**: en las series, 84 países tienen algún día con acumulado que baja;
en los reportes son 159. Al 9-mar-2023 ambas fuentes coinciden (en casos todos los países; en fallecidos, todos menos uno),
pero en el camino no:
Johns Hopkins corrigió **hacia atrás** las series y dejó los reportes ya publicados como estaban.

In [12]:
URL_SERIE = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/"
             "csse_covid_19_time_series/time_series_covid19_{}_global.csv")

difs = {}
for serie, col in [("confirmed", "confirmed"), ("deaths", "deaths")]:
    ancha = pd.read_csv(URL_SERIE.format(serie))
    larga = ancha.drop(columns=["Province/State", "Lat", "Long"]).melt(
        id_vars="Country/Region", var_name="fecha_reporte", value_name="serie")
    larga["fecha_reporte"] = pd.to_datetime(larga["fecha_reporte"], format="%m/%d/%y")
    larga = (larga.rename(columns={"Country/Region": "country_region"})
                  .groupby(["country_region", "fecha_reporte"], as_index=False)["serie"].sum())

    # Retrocesos dentro de la serie (el 84 de la PPT)
    baja = larga.groupby("country_region")["serie"].diff() < 0
    print(f"Serie {serie}: países con algún día que baja:", larga.loc[baja, "country_region"].nunique())

    c = pais.merge(larga, on=["country_region", "fecha_reporte"])
    c["diferencia"] = c["serie"] - c[col]
    distinto = c[c["diferencia"] != 0]
    ultimo = c[c["fecha_reporte"] == c["fecha_reporte"].max()]
    print(f"  cruzan {len(c)} de {len(pais)} días país | días con diferencia: {len(distinto)}"
          f" | países: {distinto['country_region'].nunique()} | distintos al 9-mar-2023: {(ultimo['diferencia'] != 0).sum()}")
    resumen.append([f"Serie ≠ reportes en {serie} (días país)", len(distinto), distinto["country_region"].nunique()])
    difs[serie] = distinto

(difs["confirmed"].groupby("country_region")["diferencia"]
    .agg(dias="size", mayor_diferencia=lambda x: x.abs().max())
    .nlargest(10, "dias"))

Serie confirmed: países con algún día que baja: 84
  cruzan 220594 de 220594 días país | días con diferencia: 4089 | países: 165 | distintos al 9-mar-2023: 0


Serie deaths: países con algún día que baja: 80
  cruzan 220594 de 220594 días país | días con diferencia: 2059 | países: 111 | distintos al 9-mar-2023: 1


,dias,mayor_diferencia
country_region,,
US,1097,391197.0
Germany,756,83957.0
Netherlands,726,380388.0
Sweden,184,3463.0
United Kingdom,159,39329.0
France,128,87784.0
Japan,112,85781.0
Chile,109,34279.0
Pakistan,96,5834.0


### Resumen

In [13]:
pd.DataFrame(resumen, columns=["problema", "número", "países afectados"])

,problema,número,países afectados
0,Acumulado de casos que baja (días país),350,159
1,Acumulado de fallecidos que baja (días país),272,111
2,Días sin reporte (acumulado sin cambio),69604,200
3,Días en que el país falta en el archivo,175,10
4,Picos por acumulación de rezagos,2799,181
5,Fallecidos > casos (filas de territorio),10777,10
6,Fallecidos > casos (días país),302,3
7,Serie ≠ reportes en confirmed (días país),4089,165
8,Serie ≠ reportes en deaths (días país),2059,111


## 4 · Índice de calidad por país y decisiones (H2.4)

### 4.1 · Índice de calidad por país
Seis indicadores por país, cada uno como **proporción de sus días** (o de sus filas) con el problema de la sección 3:

| Indicador | Viene de |
|---|---|
| `retrocesos` | 3.1 · acumulado de casos o fallecidos que baja |
| `sin_reporte` | 3.2 · acumulado sin cambio |
| `ausente` | 3.2 · días en que el país falta en el archivo |
| `picos` | 3.3 · picos por acumulación de rezagos |
| `fallecidos_mayor` | 3.4 · filas de territorio con fallecidos > casos |
| `serie_distinta` | 3.5 · días en que la serie no coincide con los reportes |

Cada indicador se divide por su máximo entre países (queda entre 0 y 1) y el índice es
**100 × (1 − promedio de los 6)**: 100 = sin problemas. Todos pesan lo mismo.

Antes se sacan las 5 entidades que no son país (decisión de 4.2): si no, dominarían el máximo de cada indicador.
El índice mide los datos **tal como vienen**, antes de las correcciones de 4.2.

In [14]:
ENTIDADES = ["Diamond Princess", "MS Zaandam", "Summer Olympics 2020", "Winter Olympics 2022", "Antarctica"]

por_pais = pais.groupby("country_region")
indicadores = pd.DataFrame({
    "retrocesos": (pais["retrocede_casos"] | pais["retrocede_fallecidos"]).groupby(pais["country_region"]).mean(),
    "sin_reporte": sin_reporte.groupby(pais["country_region"]).mean(),
    "ausente": dias["faltan"] / (dias["count"] + dias["faltan"]),
    "picos": pais["pico_rezago"].groupby(pais["country_region"]).mean(),
    "fallecidos_mayor": f.groupby(df["country_region"]).mean(),
    "serie_distinta": difs["confirmed"].groupby("country_region").size() / por_pais.size(),
}).fillna(0)                                   # NaN = el país no tuvo ese problema
indicadores = indicadores.drop(index=ENTIDADES)

normalizado = indicadores / indicadores.max()
indicadores["indice_calidad"] = (100 * (1 - normalizado.mean(axis=1))).round(1)
print("Países con índice:", len(indicadores))
indicadores["indice_calidad"].describe().round(1)

Países con índice: 196


count    196.0
mean      89.6
std        7.1
min       68.3
25%       84.7
50%       90.6
75%       95.7
max       99.6
Name: indice_calidad, dtype: float64

### Ranking: los 10 países con datos menos confiables (alimenta P4)
Solo países con **1 millón de habitantes o más** (población de la tabla UID, leída desde raw).
Sin ese filtro el ranking lo llenan microestados como Tuvalu o la Santa Sede, que salen mal porque casi no tienen casos,
no porque informen mal. El índice igual queda calculado para todos.

In [15]:
URL_UID = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"
uid = pd.read_csv(URL_UID)
poblacion = uid[uid["Province_State"].isna()].set_index("Country_Region")["Population"]   # fila del país completo
indicadores["poblacion"] = poblacion

print("Países sin población en UID:", indicadores["poblacion"].isna().sum())
grandes = indicadores[indicadores["poblacion"] >= 1_000_000]
print("Países con 1 millón o más:", len(grandes))
print("Chile:", indicadores.loc["Chile", "indice_calidad"],
      "| puesto", int(grandes["indice_calidad"].rank(ascending=False)["Chile"]), "de", len(grandes), "(1 = mejor)")

peores = grandes.nsmallest(10, "indice_calidad")
peores.round(3)

Países sin población en UID: 0
Países con 1 millón o más: 157
Chile: 96.6 | puesto 34 de 157 (1 = mejor)


,retrocesos,sin_reporte,ausente,picos,fallecidos_mayor,serie_distinta,indice_calidad,poblacion
country_region,,,,,,,,
Colombia,0.001,0.257,0.037,0.034,0.015,0.062,68.3,50882884.0
Botswana,0.003,0.778,0.000,0.064,0.000,0.001,69.1,2351625.0
Germany,0.043,0.112,0.000,0.003,0.000,0.665,69.2,83155031.0
Cameroon,0.001,0.812,0.000,0.063,0.000,0.001,69.6,26545864.0
Sweden,0.007,0.536,0.000,0.049,0.000,0.162,72.6,10099270.0
Mexico,0.005,0.182,0.031,0.026,0.000,0.040,73.8,127792286.0
Mauritius,0.000,0.646,0.000,0.053,0.000,0.019,74.9,1271767.0
Congo (Brazzaville),0.002,0.839,0.000,0.039,0.000,0.000,75.1,5518092.0
Switzerland,0.016,0.420,0.000,0.044,0.000,0.001,75.1,8654618.0


### 4.2 · Decisiones
Cada problema tiene una decisión: **corregir** (hay una fuente mejor o una regla segura), **marcar** (se deja el dato
y una columna avisa) o **excluir** (no sirve para lo que se analiza). Imputar es inventar: no se rellena nada.

| Problema | Filas o días | Decisión | Cómo |
|---|---|---|---|
| Territorio × día repetido | 156 filas | Corregir | Sección 1: se deja la fila de mayor `confirmed` |
| Acumulado negativo | 6 filas | Corregir a vacío | `NaN` en el acumulado y en los nuevos de ese día y del siguiente: no hay de dónde sacar el valor real |
| Acumulado que baja | 350 días país | Marcar | `retrocede_casos`, `retrocede_fallecidos` (sección 1) |
| Días sin reporte | 69.604 días país | Marcar | `sin_reporte`: acumulado > 0 sin cambio respecto al día anterior |
| Picos de rezago | 2.799 días país | Marcar | `pico_rezago`: no se redistribuyen los casos, no sabemos a qué días corresponden |
| Fallecidos > casos | 10.777 filas | Marcar | Se mantienen: son filas `Unassigned` que suman al total del país; la regla de consistencia de H2.2 las avisa |
| Series ≠ reportes | 4.089 días país | Fuente única | Se usan los reportes diarios; las series quedan solo como control (al 9-mar-2023 coinciden) |
| Entidades que no son país | 5 | Excluir | Diamond Princess, MS Zaandam, Summer Olympics 2020, Winter Olympics 2022, Antarctica: sin población, no admiten tasas por 100.000 |
| Cruceros y repatriados dentro de países | ver abajo | Excluir | Provincias con «Princess» (Grand/Diamond Princess en EE.UU. y Canadá, «From Diamond Princess») y `Repatriated Travellers` |
| Resto de territorios sin `uid` | ~0,05 % | Marcar | `uid` queda vacío; se mantienen porque suman al total del país (ciudades EE.UU./Canadá feb–mar 2020, `New York City`, `Alaska/Yakutat`, `Taiwan*/Taiwan`) |
| `recovered` y `active` | — | No se usan | Dejan de actualizarse en ago-2021; se documentan en el diccionario |

Las marcas se calculan a nivel **territorio** con la misma definición de la sección 3
(a nivel país se vuelven a obtener sumando territorios).

In [16]:
# Excluir entidades que no son país, cruceros y repatriados
excluir = (df["country_region"].isin(ENTIDADES)
           | df["province_state"].str.contains("Princess")
           | (df["province_state"] == "Repatriated Travellers"))
print("Filas excluidas:", excluir.sum())
df = df[~excluir].reset_index(drop=True)

# Acumulados negativos -> NaN (y los nuevos de ese día y del siguiente, que se calcularon con ese valor)
for acumulado, nuevos, marca in [("confirmed", "casos_nuevos", "retrocede_casos"),
                                 ("deaths", "fallecidos_nuevos", "retrocede_fallecidos")]:
    negativo = df[acumulado] < 0
    siguiente = negativo.groupby([df[c] for c in llave]).shift(1, fill_value=False)
    df.loc[negativo, acumulado] = float("nan")
    df.loc[negativo | siguiente, nuevos] = float("nan")
    df[marca] = df[nuevos] < 0
    print(f"{acumulado} negativo -> NaN:", negativo.sum())

# Marcas nuevas a nivel territorio
df["sin_reporte"] = (df["confirmed"] > 0) & (df["casos_nuevos"] == 0)
media_7_previa = df.groupby(llave)["casos_nuevos"].transform(lambda s: s.shift(1).rolling(7).mean())
df["pico_rezago"] = (df["casos_nuevos"] > 5 * media_7_previa) & (df["casos_nuevos"] >= 100)

print("Filas que quedan:", len(df))
print("Marcadas · retrocede casos:", df["retrocede_casos"].sum(), "| sin reporte:", df["sin_reporte"].sum(),
      "| pico de rezago:", df["pico_rezago"].sum(), "| sin uid:", df["uid"].isna().sum())

Filas excluidas: 10918


confirmed negativo -> NaN: 4


deaths negativo -> NaN: 5


Filas que quedan: 4276477
Marcadas · retrocede casos: 44576 | sin reporte: 1740170 | pico de rezago: 37998 | sin uid: 2173


### Control
Chile no tiene filas excluidas ni negativas: sumando sus territorios por día tiene que dar
exactamente el mismo acumulado que la tabla `pais` de la sección 1, y terminar en 5.192.286.

In [17]:
chile_dia = df[df["country_region"] == "Chile"].groupby("fecha_reporte")["confirmed"].sum()
chile_antes = pais[pais["country_region"] == "Chile"].set_index("fecha_reporte")["confirmed"]
print("Chile · mismo acumulado día a día que en la sección 1:", chile_dia.equals(chile_antes),
      "| último:", int(chile_dia.iloc[-1]))
print("Países que quedan:", df["country_region"].nunique())
reglas_calidad(df)

Chile · mismo acumulado día a día que en la sección 1: True | último: 5192286
Países que quedan: 196


,dimensión,regla,filas,%,umbral %,estado
0,Completitud,confirmed ausente,32,0.00,1.0,OK
1,Completitud,territorio sin uid (no cruza con UID),2173,0.05,1.0,OK
2,Unicidad,territorio × día repetido,0,0.00,0.0,OK
3,Validez,confirmed o deaths negativo,0,0.00,0.0,OK
4,Validez,lat o long fuera de rango,0,0.00,0.0,OK
5,Validez,fecha_reporte fuera de 22-ene-2020 → 9-mar-2023,0,0.00,0.0,OK
6,Consistencia,fallecidos mayores que casos,9734,0.23,0.1,ALERTA
7,Exactitud,acumulado de casos retrocede,44576,1.04,1.0,ALERTA
8,Exactitud,acumulado de fallecidos retrocede,16181,0.38,1.0,OK
9,Oportunidad,last_update con más de 7 días de atraso,11621,0.27,5.0,OK


### 4.3 · Guardar
- `data/clean/covid_diario.parquet`: territorio × día con casos y fallecidos nuevos y todas las marcas. Es lo que
  leen el DW, el análisis y el tablero.
- `data/clean/indice_calidad_pais.parquet`: índice e indicadores por país (P4 y tablero).

`covid_diario` se guarda con `compression="zstd"`: sin ella superaría los 100 MB que GitHub acepta por archivo
(la tabla de `01_ingesta` ya pesaba 99 MB y aquí se agregan columnas). Zstandard comprime más que el `snappy` que usa
pandas por defecto y se lee igual de rápido; pandas y pyarrow lo leen sin instalar nada extra.

In [18]:
import os

RUTA_DIARIO = "../data/clean/covid_diario.parquet"
RUTA_INDICE = "../data/clean/indice_calidad_pais.parquet"

df.to_parquet(RUTA_DIARIO, index=False, compression="zstd")
indicadores.reset_index(names="country_region").to_parquet(RUTA_INDICE, index=False)

print(f"covid_diario: {len(df)} filas × {df.shape[1]} columnas · {os.path.getsize(RUTA_DIARIO) / 2**20:.1f} MB")
print(f"indice_calidad_pais: {len(indicadores)} países · {os.path.getsize(RUTA_INDICE) / 2**10:.1f} KB")

covid_diario: 4276477 filas × 26 columnas · 59.2 MB
indice_calidad_pais: 196 países · 15.5 KB


## 5 · Linaje (H2.6)
Cada fila trae `archivo_origen` y `fila_origen` desde `01_ingesta`: de qué CSV de JHU y de qué fila salió
(`fila_origen` = posición en `pd.read_csv`, 0 = primera fila de datos).

Un número del tablero casi nunca es una sola fila: el acumulado de un país es la **suma** de sus territorios, y los
casos nuevos son la **resta** de dos días. `linaje(pais, fecha)` lista las filas de origen detrás de ese número.

In [19]:
def linaje(pais, fecha):
    """Filas de origen (archivo y fila) que suman el acumulado de un país en una fecha."""
    filas = df[(df["country_region"] == pais) & (df["fecha_reporte"] == fecha)]
    return filas[["province_state", "admin2", "confirmed", "deaths", "archivo_origen", "fila_origen"]]


origen = linaje("Chile", "2020-06-17")
print("Filas detrás del número:", len(origen), "| acumulado de Chile el 17-jun-2020:", int(origen["confirmed"].sum()))
origen

Filas detrás del número: 17 | acumulado de Chile el 17-jun-2020: 220628


,province_state,admin2,confirmed,deaths,archivo_origen,fila_origen
95603,Antofagasta,,5241.0,81.0,06-17-2020.csv,3092
96627,Araucania,,2815.0,36.0,06-17-2020.csv,3097
97651,Arica y Parinacota,,1218.0,11.0,06-17-2020.csv,3099
98675,Atacama,,561.0,0.0,06-17-2020.csv,3106
99699,Aysen,,26.0,0.0,06-17-2020.csv,3110
100723,Biobio,,4015.0,25.0,06-17-2020.csv,3126
101747,Coquimbo,,1980.0,12.0,06-17-2020.csv,3175
102771,Los Lagos,,1249.0,14.0,06-17-2020.csv,3315
103795,Los Rios,,555.0,10.0,06-17-2020.csv,3316
104819,Magallanes,,1284.0,15.0,06-17-2020.csv,3324


### Control: volver al CSV original
Se lee desde raw el archivo que dice `archivo_origen` y se toman las filas que dice `fila_origen`:
tienen que ser Chile y sumar lo mismo. Con el día anterior se reconstruyen los casos nuevos del 17-jun (el pico de rezago).

In [20]:
URL_REPORTES = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_daily_reports/"

archivo = origen["archivo_origen"].iloc[0]
en_crudo = pd.read_csv(URL_REPORTES + archivo).iloc[origen["fila_origen"]]
print("Archivo:", archivo, "| países en esas filas:", en_crudo["Country_Region"].unique().tolist())
print("Mismos casos fila a fila:", (en_crudo["Confirmed"].values == origen["confirmed"].values).all(),
      "| suma en el CSV original:", int(en_crudo["Confirmed"].sum()))

dia_antes = linaje("Chile", "2020-06-16")
hoy, ayer = origen["confirmed"].sum(), dia_antes["confirmed"].sum()
print()
print("Casos nuevos del 17-jun =", int(hoy), "(" + archivo + ") −", int(ayer),
      "(" + dia_antes["archivo_origen"].iloc[0] + ") =", int(hoy - ayer))

Archivo: 06-17-2020.csv | países en esas filas: ['Chile']
Mismos casos fila a fila: True | suma en el CSV original: 220628

Casos nuevos del 17-jun = 220628 (06-17-2020.csv) − 184449 (06-16-2020.csv) = 36179
